# MOFA+ — Multi-Dataset Methylation Integration
## Step 3: Finding shared CpG signatures across all 9 exposure cohorts

**What MOFA+ does:**
MOFA+ (Multi-Omics Factor Analysis v2) learns *latent factors* — hidden axes of CpG co-variation — that are shared across multiple datasets. Instead of asking "does wildfire change methylation?" (one dataset, low power), it asks: **"which CpG patterns vary consistently across wildfire, stress, obesity, and cfDNA cohorts?"**

**Our design:**
- **Views (2):** `cpgi` and `genebody` — two biological compartments treated as separate data types
- **Groups (9):** one per cohort (wildfire, stress, obesity × 3, cfDNA × 4)
- **Features:** CpG positions — shared reference, `NaN` where a dataset has no coverage
- **Samples:** individual animals within each group

**Why this beats running models separately per dataset:**
Each cohort has n=11–21 (low power). MOFA+ borrows statistical strength across all groups simultaneously. A factor that separates Case/Control in 5 of 9 cohorts is compelling even if no single cohort reaches significance.

**MOFA+ handles missing data natively** — datasets that do not cover a CpG simply contribute `NaN`; the model estimates weights using only observed values.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
from pathlib import Path
from scipy import stats

from mofapy2.run.entry_point import entry_point

warnings.filterwarnings('ignore')
print('Libraries loaded.')

## Cell 2 — Configuration

**What to remember:**
- `N_FACTORS` controls model complexity. Start with 5 — more factors = more patterns captured but slower and harder to interpret. You can re-run with 8–10 once you know the data.
- `MIN_COVERAGE` drops CpGs that are missing in too many groups — a fully-missing CpG contributes nothing to MOFA+.
- We run two views (`cpgi`, `genebody`) as separate inputs so MOFA+ can tell us whether a factor is driven by island methylation, gene body methylation, or both.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd().parent
DATA_DIR     = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR  = PROJECT_ROOT / 'results' / 'figures'
TABLES_DIR   = PROJECT_ROOT / 'results' / 'tables'
MOFA_DIR     = PROJECT_ROOT / 'results' / 'mofa'

for d in [FIGURES_DIR, TABLES_DIR, MOFA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── MOFA+ settings ────────────────────────────────────────────────────
N_FACTORS    = 5     # number of latent factors to learn
N_ITER       = 1000  # training iterations (increase to 2000 if not converged)
MIN_COVERAGE = 0.5   # drop CpGs missing in more than 50% of groups

# ── All datasets and regions ──────────────────────────────────────────
DATASETS = [
    'wildfire',
    'stress',
    'obesity_hippocampus',
    'obesity_hypothalamus',
    'obesity_prefrontalcortex',
    'cfdna_GD45',
    'cfdna_GD90',
    'cfdna_GD120',
    'cfdna_GD150',
]
REGIONS = ['cpgi', 'genebody']

print(f'N_FACTORS    : {N_FACTORS}')
print(f'N_ITER       : {N_ITER}')
print(f'MIN_COVERAGE : {MIN_COVERAGE} (drop CpG if missing in >{MIN_COVERAGE*100:.0f}% of groups)')

## Cell 3 — Load and Align All Beta Matrices

**What alignment means:**
Each dataset was preprocessed independently and may cover a slightly different set of CpGs (depending on sequencing depth). MOFA+ requires all groups within a view to have the **same feature columns**. We solve this by:
1. Taking the **union** of all CpG positions across all datasets for each region
2. **Reindexing** each dataset's matrix to the full union — CpGs not covered by that dataset become `NaN`
3. Dropping CpGs missing in more than `MIN_COVERAGE` fraction of groups (they contribute no cross-cohort information)

**`df.reindex(columns=all_cpgs)`** adds the missing columns as `NaN` without changing existing values. It also reorders columns to match the reference set.

In [ ]:
view_data   = {}   # view_data[region][dataset] = DataFrame(samples × CpGs)
available   = {}   # track which files exist

for region in REGIONS:
    matrices   = {}
    for ds in DATASETS:
        path = DATA_DIR / f'{ds}_{region}_methylation.csv'
        if path.exists():
            df = pd.read_csv(path, index_col=0).T   # transpose: samples × CpGs
            # Ensure sample index is string (avoids int/str mismatch)
            df.index = df.index.astype(str)
            matrices[ds] = df
            print(f'  Loaded {ds:30s} {region}: {df.shape[0]:3d} samples × {df.shape[1]:3d} CpGs')
        else:
            print(f'  [MISSING] {ds}_{region}_methylation.csv — skipping')

    # ── Union of all CpG positions ──────────────────────────────────
    all_cpgs = sorted(set().union(*[set(df.columns) for df in matrices.values()]))
    print(f'\n  {region}: {len(all_cpgs)} CpGs in union across {len(matrices)} datasets')

    # ── Reindex every dataset to full union (NaN for missing CpGs) ──
    for ds in matrices:
        matrices[ds] = matrices[ds].reindex(columns=all_cpgs)

    # ── Drop CpGs missing in too many groups ─────────────────────────
    # For each CpG, count how many groups have at least 1 non-NaN value
    n_groups    = len(matrices)
    cpg_coverage = pd.DataFrame({
        ds: (~matrices[ds].isna().all(axis=0)).astype(int)
        for ds in matrices
    })
    # cpg_coverage: rows=CpGs, columns=datasets, value=1 if CpG observed in that dataset
    fraction_covered = cpg_coverage.sum(axis=1) / n_groups
    keep_cpgs = fraction_covered[fraction_covered >= MIN_COVERAGE].index
    print(f'  Kept {len(keep_cpgs)}/{len(all_cpgs)} CpGs (coverage >= {MIN_COVERAGE*100:.0f}% of groups)')

    for ds in matrices:
        matrices[ds] = matrices[ds][keep_cpgs]

    view_data[region] = matrices
    available[region] = list(matrices.keys())
    print()

print('Data loading complete.')

## Cell 4 — Run MOFA+

**Understanding the model setup:**

`set_data_options(scale_views=False)` — we do NOT scale because both views are already beta values (0–1 range). Scaling would be needed if one view were, say, RNA-seq counts and another were methylation.

`set_model_options(factors=N_FACTORS)` — MOFA+ will learn this many factors. Each factor is a direction in CpG space that explains variance across groups.

`likelihoods=['gaussian', 'gaussian']` — assumes normally distributed noise around factor predictions. Appropriate for beta values (especially after mean imputation keeps most values in the interior of [0,1]).

**Training output to watch:**
- ELBO (Evidence Lower BOund) should increase monotonically — if it decreases, something is wrong
- "Converged" message means the model stopped early because improvement was below threshold

In [ ]:
# ── Build MOFA+ data structure ────────────────────────────────────────
# data_dict[view][group] = DataFrame(n_samples × n_features)
# Groups must have same columns within a view; can differ across views
data_dict = {}
for region in REGIONS:
    data_dict[region] = {}
    for ds in available[region]:
        data_dict[region][ds] = view_data[region][ds]
        # MOFA+ tolerates NaN within a matrix (missing sample/CpG combinations)

# ── Run MOFA+ ─────────────────────────────────────────────────────────
print('Setting up MOFA+ model...')
ent = entry_point()

ent.set_data_matrix(
    data_dict,
    likelihoods=['gaussian'] * len(REGIONS)
    # One likelihood per VIEW, not per group
)

ent.set_data_options(
    scale_views=False,   # all views already on same 0–1 beta scale
    scale_groups=False   # do not standardise across groups
)

ent.set_model_options(
    factors=N_FACTORS
)

ent.set_train_options(
    iter=N_ITER,
    convergence_mode='fast',   # 'medium' or 'slow' for final analysis
    seed=42,
    verbose=False,
    outfile=str(MOFA_DIR / 'mofa_model.hdf5')
)

print('Building model...')
ent.build()

print(f'Training MOFA+ with {N_FACTORS} factors × {len(REGIONS)} views × {len(DATASETS)} groups...')
ent.run()

print('\nMOFA+ training complete.')

## Cell 5 — Extract Results

After training, MOFA+ stores everything in an HDF5 file. We extract:
- **Z (factor scores):** one matrix per group, shape `(n_samples, n_factors)` — where each animal sits in factor space
- **W (weights/loadings):** one matrix per view, shape `(n_cpgs, n_factors)` — how much each CpG contributes to each factor
- **R² (variance explained):** how much variance each factor explains per view and per group

In [ ]:
# ── Load results from saved HDF5 ──────────────────────────────────────
mofa_file = MOFA_DIR / 'mofa_model.hdf5'

# Extract factor scores (Z) and weights (W)
factor_scores = {}   # factor_scores[dataset] = DataFrame(samples × Factor1..N)
cpg_weights   = {}   # cpg_weights[region]   = DataFrame(CpGs × Factor1..N)

factor_cols = [f'Factor{i+1}' for i in range(N_FACTORS)]

with h5py.File(mofa_file, 'r') as f:

    groups = list(f['expectations']['Z'].keys())
    views  = list(f['expectations']['W'].keys())
    print(f'Groups in model : {groups}')
    print(f'Views  in model : {views}')

    # Factor scores per group
    for grp in groups:
        Z = f['expectations']['Z'][grp][:]   # shape: (n_factors, n_samples)
        # HDF5 stores as factors × samples — transpose to samples × factors
        Z = Z.T
        # Sample names
        samples = [s.decode() if isinstance(s, bytes) else s
                   for s in f['samples'][grp][:]]
        factor_scores[grp] = pd.DataFrame(Z, index=samples, columns=factor_cols)

    # CpG weights per view
    for view in views:
        W = f['expectations']['W'][view][:]   # shape: (n_factors, n_cpgs)
        W = W.T                               # → n_cpgs × n_factors
        features = [feat.decode() if isinstance(feat, bytes) else feat
                    for feat in f['features'][view][:]]
        cpg_weights[view] = pd.DataFrame(W, index=features, columns=factor_cols)

    # Variance explained
    r2_dict = {}
    for view in views:
        r2_dict[view] = {}
        for grp in groups:
            try:
                r2 = f['variance_explained']['r2_per_factor'][view][grp][:]
                r2_dict[view][grp] = r2
            except KeyError:
                pass

print('\nFactor score shapes:')
for grp, df in factor_scores.items():
    print(f'  {grp:35s}: {df.shape}')
print('\nCpG weight shapes:')
for view, df in cpg_weights.items():
    print(f'  {view:15s}: {df.shape}')

## Cell 6 — Variance Explained

The variance explained heatmap shows which factors matter most in which cohorts and which view (cpgi vs genebody). This is the first diagnostic plot — it tells you:
- **Bright cells** = that factor explains a lot of variation in that group
- **Factors that light up across many groups** = shared biological signal
- **Factors that light up in only one group** = cohort-specific effect (still interesting!)

In [ ]:
# ── Variance explained heatmap ────────────────────────────────────────
fig, axes = plt.subplots(1, len(REGIONS), figsize=(6 * len(REGIONS), 6), squeeze=False)

for col_idx, region in enumerate(REGIONS):
    if region not in r2_dict or not r2_dict[region]:
        axes[0, col_idx].set_title(f'{region} — no data')
        continue

    # Build matrix: rows=groups, columns=factors
    r2_matrix = pd.DataFrame(r2_dict[region]).T   # groups × factors
    r2_matrix.columns = factor_cols[:r2_matrix.shape[1]]

    ax = axes[0, col_idx]
    sns.heatmap(
        r2_matrix * 100,           # convert to percentage
        annot=True, fmt='.1f',
        cmap='YlOrRd',
        vmin=0, vmax=None,
        linewidths=0.5,
        ax=ax,
        cbar_kws={'label': 'Variance explained (%)'}
    )
    ax.set_title(f'Variance Explained — {region}', fontsize=12)
    ax.set_xlabel('Factor')
    ax.set_ylabel('Dataset / Group')

plt.suptitle('MOFA+ Variance Explained per Factor', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'mofa_variance_explained.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: mofa_variance_explained.png')

## Cell 7 — Factor Scores Coloured by Case/Control

For each dataset and each factor, we plot the factor scores split by case vs control. A factor that consistently separates case from control across multiple cohorts is the most important finding — it represents a shared methylation signature of early-life exposure.

**How to read these plots:**
- X-axis: factor score for each animal
- Groups: Control (blue) vs Case (red)
- If the distributions separate → this factor captures exposure-related variation in that cohort

In [ ]:
# ── Load labels for all datasets ─────────────────────────────────────
all_labels = {}
for ds in DATASETS:
    lpath = DATA_DIR / f'{ds}_labels.csv'
    if lpath.exists():
        lab = pd.read_csv(lpath, index_col='sample_id')
        lab.index = lab.index.astype(str)
        all_labels[ds] = lab['label']

# ── Factor score strip plots ──────────────────────────────────────────
n_groups_avail = len(factor_scores)
fig, axes = plt.subplots(
    N_FACTORS, n_groups_avail,
    figsize=(3 * n_groups_avail, 2.5 * N_FACTORS),
    squeeze=False
)

group_list = list(factor_scores.keys())
colors     = {0: 'steelblue', 1: 'tomato'}

for fi, factor in enumerate(factor_cols):
    for gi, grp in enumerate(group_list):
        ax    = axes[fi, gi]
        scores = factor_scores[grp][factor]

        if grp in all_labels:
            y = all_labels[grp].reindex(scores.index)
            for val, color in colors.items():
                mask = y == val
                ax.scatter(
                    [val] * mask.sum(),
                    scores[mask],
                    c=color, alpha=0.7, s=30, zorder=3
                )
            ax.set_xticks([0, 1])
            ax.set_xticklabels(['Ctrl', 'Case'], fontsize=7)
        else:
            ax.plot(scores.values, 'o', color='grey', ms=4)

        ax.set_ylabel(factor if gi == 0 else '', fontsize=8)
        ax.set_title(grp.replace('_', '\n') if fi == 0 else '', fontsize=7)
        ax.axhline(0, color='grey', lw=0.5, ls='--')

plt.suptitle('MOFA+ Factor Scores (blue=control, red=case)', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'mofa_factor_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: mofa_factor_scores.png')

## Cell 8 — Top CpG Weights per Factor

The weights tell you **which CpGs drive each factor**. High positive weight = CpG is more methylated in samples with high factor scores. High negative weight = CpG is less methylated.

These are your candidate CpGs for the per-CpG meta-analysis in the next step. A CpG that has a high weight in a factor that also separates case/control is your primary hit.

In [ ]:
# ── Top CpG weights per factor per view ──────────────────────────────
N_TOP = 10  # top CpGs to show per factor

for region in REGIONS:
    if region not in cpg_weights:
        continue
    W = cpg_weights[region]
    print(f'\n── {region.upper()} — Top {N_TOP} CpGs per Factor ──')
    for factor in factor_cols:
        top = W[factor].abs().sort_values(ascending=False).head(N_TOP)
        print(f'\n  {factor}:')
        for cpg, weight in top.items():
            direction = '+' if W.loc[cpg, factor] > 0 else '-'
            print(f'    {direction}{weight:.4f}  {cpg}')

# ── Save weights ──────────────────────────────────────────────────────
for region in REGIONS:
    if region in cpg_weights:
        cpg_weights[region].to_csv(TABLES_DIR / f'mofa_weights_{region}.csv')
        print(f'\nSaved: mofa_weights_{region}.csv')

for grp in factor_scores:
    factor_scores[grp].to_csv(TABLES_DIR / f'mofa_scores_{grp}.csv')
print('Saved: mofa_scores_*.csv for all groups')

## Cell 9 — Factor–Label Association (Mann-Whitney U test)

For each factor × group combination where we have case/control labels, we test whether factor scores differ significantly between groups using a Mann-Whitney U test (non-parametric, appropriate for small n).

**What to look for:**
- Low p-values (< 0.05) after correction = factor is associated with exposure in that cohort
- If the same factor is significant in 2+ cohorts → strongest evidence of a shared signature

**Bonferroni correction** divides the significance threshold by the number of tests. It is conservative (may miss real effects) but appropriate as a first pass.

In [ ]:
from scipy.stats import mannwhitneyu

results = []
n_tests = 0

for factor in factor_cols:
    for grp in factor_scores:
        if grp not in all_labels:
            continue
        scores = factor_scores[grp][factor]
        y      = all_labels[grp].reindex(scores.index).dropna()
        scores = scores.reindex(y.index)

        ctrl = scores[y == 0]
        case = scores[y == 1]

        if len(ctrl) < 3 or len(case) < 3:
            # Too few samples for a meaningful test
            continue

        stat, pval = mannwhitneyu(ctrl, case, alternative='two-sided')
        n_tests   += 1
        results.append({
            'factor'       : factor,
            'dataset'      : grp,
            'n_ctrl'       : len(ctrl),
            'n_case'       : len(case),
            'median_ctrl'  : round(ctrl.median(), 4),
            'median_case'  : round(case.median(), 4),
            'MW_stat'      : round(stat, 2),
            'p_value'      : round(pval, 4),
        })

results_df = pd.DataFrame(results).sort_values('p_value')

# Bonferroni correction
alpha_corrected = 0.05 / max(n_tests, 1)
results_df['significant_bonferroni'] = results_df['p_value'] < alpha_corrected

print(f'Total tests: {n_tests}')
print(f'Bonferroni threshold: p < {alpha_corrected:.4f}')
print()
print('── Top associations (sorted by p-value) ──')
display(results_df.head(20))

results_df.to_csv(TABLES_DIR / 'mofa_factor_label_association.csv', index=False)
print('\nSaved: mofa_factor_label_association.csv')